In [1]:
!wget https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip
!unzip hetrec2011-lastfm-2k.zip

--2026-05-31 06:30:56--  https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2589075 (2.5M) [application/zip]
Saving to: ‘hetrec2011-lastfm-2k.zip’

hetrec2011-lastfm-2 100%[===================>]   2.47M  5.58MB/s    in 0.4s    

2026-05-31 06:30:57 (5.58 MB/s) - ‘hetrec2011-lastfm-2k.zip’ saved [2589075/2589075]

Archive:  hetrec2011-lastfm-2k.zip
  inflating: user_friends.dat        
  inflating: user_taggedartists.dat  
  inflating: user_taggedartists-timestamps.dat  
  inflating: artists.dat             
  inflating: readme.txt              
  inflating: tags.dat                
  inflating: user_artists.dat        


In [17]:
import numpy as np
from scipy.sparse import csc_matrix
# ElasticNet : L1, L2 규제를 포함하는 선형회귀 모델
# SLIM 모델의 각 item representation 학습에 활용
from sklearn.linear_model import ElasticNet
from tqdm import tqdm

In [6]:
artists = {} # 아티스트 정보 저장을 위한 딕셔너리 초기화

with open('artists.dat', 'r') as f:
    print(f.readline()) # 헤더 라인 출력

    # 파일의 각 라인 반복
    for line in f:
        # 탭으로 라인 분할하여 아티스트 ID, 이름 추출
        id, name, _, _ = line.split('\t')
        # 딕셔너리에 아티스트 ID와 이름 저장
        artists[int(id)] = name

id	name	url	pictureURL



In [7]:
users = [] # 사용자 ID 리스트
items = [] # 아이템(아티스트) ID 리스트
counts = [] # 상호작용 횟수(재생 횟수) 리스트

# user_artists.dat: 유저가 아티스트의 곡을 들은 횟수 정보
with open('user_artists.dat', 'r') as f:
    print(f.readline())

    for line in f:
        # 탭으로 라인 분할하여 사용자 ID, 아티스트 ID, 재생 횟수 추출
        uid, mid, count = line.strip().split('\t')

        # 파싱된 값들을 리스트에 추가 (재생 횟수는 로그 변환하여 스무딩)
        users.append(int(uid))
        items.append(int(mid))
        counts.append(np.log(1 + float(count)))

users = np.array(users, dtype=np.int32)
items = np.array(items, dtype=np.int32)
counts = np.array(counts)

userID	artistID	weight



# SLIM using ElasticNet

In [8]:
n_users = max(users) + 1
n_items = max(items) + 1

# 상호작용 데이터로 CSC(Compressed Sparse Column) 희소 행렬 생성 (학습 데이터)
train = csc_matrix((counts, (users, items)))

print(train.shape) # 희소 행렬의 모양(행, 열) 출력
print(train.nnz) # 희소 행렬의 0이 아닌 요소 수 출력

(2101, 18746)
92834


In [9]:
# SLIM을 위한 ElasticNet 모델 초기화
model = ElasticNet(
    alpha=0.1, # 규제 강도
    l1_ratio=0.5, # L1, L2 규제 혼합 비율
    positive=True, # 계수를 양수로 강제
    fit_intercept=False, # 절편 계산 안함
    copy_X=False, # X 복사 여부
    selection='random', # 특성 업데이트 방식
    tol=1e-6, # 중단 기준 허용 오차
    max_iter=100 # 최대 반복 횟수
)

In [10]:
rows, cols, data = [], [], [] # 유사도 행렬 W를 위한 리스트 초기화

for current_item in tqdm(range(n_items)):

    # 현재 아이템의 모든 사용자 상호작용 횟수 가져오기
    y = train[:, current_item].toarray().ravel()

    # 상호작용이 없으면 건너뛰기
    if np.all(y == 0):
        continue

    # 현재 아이템 데이터의 시작점과 끝점 가져오기
    s = train.indptr[current_item]
    e = train.indptr[current_item + 1]

    # 자기 예측 방지를 위해 현재 아이템의 상호작용을 임시로 0으로 설정
    backup = train.data[s:e].copy() # 원본 데이터 백업
    train.data[s:e] = 0.0 # 현재 아이템 데이터 0으로 설정

    # ElasticNet 모델 학습: 다른 아이템의 상호작용으로 현재 아이템의 상호작용 예측
    model.fit(train, y)

    # 학습된 모델에서 희소 계수(유사도) 가져오기
    coef = model.sparse_coef_

    # 계수(데이터)와 관련 아이템 ID(인덱스)를 리스트에 추가
    rows.extend(coef.indices)
    cols.extend([current_item] * coef.getnnz()) # 현재 아이템 ID 반복
    data.extend(coef.data)

    # 현재 아이템의 원래 상호작용 데이터 복원
    train.data[s:e] = backup

# 수집된 데이터로 유사도 행렬 W 구성
# W[i, j]는 아이템 j가 아이템 i에 미치는 영향을 나타냄
W = csc_matrix(
    (data, (rows, cols)),
    shape=(n_items, n_items) # 행렬의 모양은 (아이템 수, 아이템 수)
)

print(W.shape) # 유사도 행렬의 모양 출력
print(W.nnz) # 유사도 행렬의 0이 아닌 요소 수 출력

100%|██████████| 18746/18746 [03:37<00:00, 86.04it/s] 

(18746, 18746)
30859


In [11]:
# 예시
# 1918 동방신기

q = 1918 # 유사한 아티스트를 찾을 아티스트 ID 설정 (예: 동방신기)

# 유사도 행렬 W에서 현재 아티스트 데이터의 시작점과 끝점 가져오기
s = W.indptr[q]
e = W.indptr[q + 1]

# W 행렬에서 유사도 점수와 유사한 아티스트 ID 추출
scores = W.data[s:e]
ids = W.indices[s:e]

# 유사도 점수를 기준으로 유사한 아티스트 ID를 내림차순으로 정렬
ids_sorted = ids[scores.argsort()[::-1]]

# 쿼리된 아티스트 이름 출력
print(f"[{artists[q]}]와 유사한 artists:")

# 정렬된 유사 아티스트 ID를 반복하여 이름 출력
for id in ids_sorted:
    print(f"- {artists[id]}")

[동방신기]와 유사한 artists:
- 소녀시대
- Super Junior
- BoA
- Brown Eyed Girls
- f(x)
- Kylie Minogue
- Christina Aguilera
- 倖田來未


In [12]:
# 예시
# 312 한국인

uid = 312 # 추천을 생성할 사용자 ID 설정

# 지정된 사용자의 모든 아티스트 상호작용 횟수 가져오기
user_counts = train[uid, :].tocsr()

# 사용자 상호작용 벡터에 유사도 행렬 W를 곱하여 추천 점수 계산
scores = user_counts.dot(W)

order = scores.data.argsort()[::-1]

k = 10

for o in order:

    artist_id = scores.indices[o] # 현재 추천 아티스트 ID
    score = scores.data[o] # 현재 추천 점수

    # 추천 아티스트가 사용자가 이미 들었던 아티스트 목록에 없으면
    if artist_id not in user_counts.indices:
        # 아티스트 ID, 이름, 추천 점수 출력
        print(artist_id, artists[artist_id], score)

        k -= 1 # 남은 추천 수 감소

    # k가 0이 되면, 충분한 새 추천을 찾았으므로 반복 중단
    if k == 0:
        break

374 宇多田ヒカル 3.6422389947974447
1904 SHINee 2.9691049555863094
2091 4minute 2.113920580106585
2101 Capsule 1.781384488411428
289 Britney Spears 1.7016532432617018
464 3OH!3 1.5240927335191181
400 2NE1 1.3433204551519533
2089 티아라 1.1964753283444585
288 Rihanna 1.0337621143827993
300 Katy Perry 0.9982990155421692
